# 178. BitNet b1.58：Ternary BitLinear、STE 与位打包怎样实现？

> **面试问题：1.58-bit 为什么是三值权重？absmean 量化、激活量化、master weight、整数计算和存储收益怎样验证？**

## 先给结论

`log2(3)≈1.585` 来自三种权重状态 `{-1,0,1}`。BitNet b1.58 训练期保留高精度 master weight，在 forward 用 absmean scale 三值化并通过 STE 回传；激活通常另行低比特量化。理论位宽不会自动变成速度，必须有紧凑打包、专用 GEMM 与端到端测量。

## 推荐回答主线

1. 区分从头量化感知训练与训练后把 FP 模型强行三值化，写出权重 scale 和 ternary code。
2. 实现 activation absmax quant、BitLinear.forward 与 STE，验证梯度更新的是 master weight。
3. 用 base-3 打包/解包证明真实存储合同，并讨论 scale/metadata 开销。
4. 比较数值误差、饱和、吞吐和质量；绑定量化 recipe、布局与 kernel。

## 教学实现边界

教学实现仍调用普通 PyTorch 浮点 matmul 来模拟反量化权重，不产生真实 ternary kernel 加速；位打包仅演示存储编码，不处理 SIMD 对齐和分块 scale。

## 一手资料

- [BitNet](https://arxiv.org/abs/2310.11453)
- [BitNet b1.58](https://arxiv.org/abs/2402.17764)
- [bitnet.cpp](https://arxiv.org/abs/2502.11880)


In [ ]:
import hashlib  # 导入本单元需要的依赖。
import json  # 导入本单元需要的依赖。
import math  # 导入本单元需要的依赖。
from dataclasses import asdict, dataclass  # 导入本单元需要的依赖。

import numpy as np  # 导入本单元需要的依赖。
import warnings  # 导入本单元需要的依赖。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import torch  # 导入本单元需要的依赖。
from torch import nn  # 导入本单元需要的依赖。

# 小矩阵按每个输出通道的 5 个输入权重分组，便于核对 scale 与整数点积。
torch.manual_seed(178)  # 计算并保存当前步骤的中间状态。
GROUP_SIZE = 5  # 计算并保存当前步骤的中间状态。
weight = torch.tensor([[-2.0, -0.2, 0.0, 0.4, 1.8], [0.1, -0.1, 0.7, -0.8, 0.0]])  # 计算并保存当前步骤的中间状态。
x = torch.randn(4, 5)  # 计算并保存当前步骤的中间状态。

assert math.isclose(math.log2(3), 1.584962500721156, rel_tol=1e-12)  # 用受控断言验证关键不变量。
assert weight.shape == (2, 5)  # 用受控断言验证关键不变量。
assert x.shape[-1] == weight.shape[-1] == GROUP_SIZE  # 用受控断言验证关键不变量。


## 1. Absmean ternary：scale 与 code 分开保存

一种教学 recipe 用 `gamma=mean(abs(W))`，将 `W/gamma` round 后裁到 -1/0/1，再用 gamma 反量化。论文/实现的中心化、阈值和分组细节可能不同，必须把具体 recipe 写进制品。


In [ ]:
def ternary_quantize(weights, group_size, eps=1e-8):  # 定义本节可复用的核心函数。
    if weights.ndim != 2 or not isinstance(group_size, int) or group_size <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("weights 必须为二维且 group_size 为正整数")  # 遇到非法合同立即显式失败。
    codes_by_group, scales, dequantized_by_group = [], [], []  # 计算并保存当前步骤的中间状态。
    for start in range(0, weights.shape[-1], group_size):  # 遍历输入元素以累积或检查结果。
        group = weights[:, start:start + group_size]  # 计算并保存当前步骤的中间状态。
        scale = group.abs().mean(-1, keepdim=True).clamp_min(eps)  # 计算并保存当前步骤的中间状态。
        codes = torch.round(group / scale).clamp(-1, 1).to(torch.int8)  # 计算并保存当前步骤的中间状态。
        codes_by_group.append(codes)  # 计算并保存当前步骤的中间状态。
        scales.append(scale)  # 计算并保存当前步骤的中间状态。
        dequantized_by_group.append(codes.float() * scale)  # 计算并保存当前步骤的中间状态。
    return torch.cat(codes_by_group, -1), torch.stack(scales, 1), torch.cat(dequantized_by_group, -1)  # 返回当前分支计算出的结果。

# code 只含三值；每个输出通道/输入分组恰有一个 scale；零权重保持零。
codes, weight_scale, dequantized = ternary_quantize(weight, GROUP_SIZE)  # 计算并保存当前步骤的中间状态。
assert set(codes.unique().tolist()) <= {-1, 0, 1}  # 用受控断言验证关键不变量。
assert dequantized.shape == weight.shape  # 用受控断言验证关键不变量。
assert weight_scale.shape == (weight.shape[0], math.ceil(weight.shape[1] / GROUP_SIZE), 1)  # 用受控断言验证关键不变量。
assert torch.equal(dequantized[weight == 0], torch.zeros_like(dequantized[weight == 0]))  # 用受控断言验证关键不变量。


## 2. Activation absmax int8：权重 1.58-bit 不等于整个算子 1.58-bit

输入激活仍需表示；可按 token 用 absmax 映射到 int8，再反量化参与教学计算。真实整数/混合精度数据流、accumulator 位宽和 scale 融合决定 kernel。


In [ ]:
def quantize_activation(activation, bits=8, eps=1e-8):  # 定义本节可复用的核心函数。
    if not isinstance(bits, int) or not 2 <= bits <= 8:  # 按当前条件选择后续控制路径。
        raise ValueError("int8 容器只支持 2 到 8 bit 激活量化")  # 遇到非法合同立即显式失败。
    qmax = 2 ** (bits - 1) - 1  # 计算并保存当前步骤的中间状态。
    scale = activation.abs().amax(dim=-1, keepdim=True).clamp_min(eps) / qmax  # 计算并保存当前步骤的中间状态。
    quantized = torch.round(activation / scale).clamp(-qmax, qmax).to(torch.int8)  # 计算并保存当前步骤的中间状态。
    dequantized = quantized.float() * scale  # 计算并保存当前步骤的中间状态。
    activation_ste = activation + (dequantized - activation).detach()  # 计算并保存当前步骤的中间状态。
    return quantized, scale, activation_ste  # 返回当前分支计算出的结果。

# forward 数值为反量化值；STE backward 对每个输入元素传递单位梯度。
activation_probe = x.clone().requires_grad_(True)  # 计算并保存当前步骤的中间状态。
act_q, act_scale, act_dq = quantize_activation(activation_probe)  # 计算并保存当前步骤的中间状态。
assert act_q.dtype == torch.int8  # 用受控断言验证关键不变量。
assert act_scale.shape == (x.shape[0], 1)  # 用受控断言验证关键不变量。
assert torch.allclose(act_dq.detach(), act_q.float() * act_scale.detach())  # 用受控断言验证关键不变量。
act_dq.sum().backward()  # 计算并保存当前步骤的中间状态。
assert torch.equal(activation_probe.grad, torch.ones_like(activation_probe))  # 用受控断言验证关键不变量。

# 反例边界 fail-closed：1 bit 会产生 qmax=0，超过 8 bit 会溢出 int8，二者都必须拒绝。
invalid_bits_rejected = []  # 计算并保存当前步骤的中间状态。
for invalid_bits in (1, 9):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        quantize_activation(x, bits=invalid_bits)  # 计算并保存当前步骤的中间状态。
    except ValueError:  # 捕获预期异常并验证失败分支。
        invalid_bits_rejected.append(invalid_bits)  # 计算并保存当前步骤的中间状态。
assert invalid_bits_rejected == [1, 9]  # 用受控断言验证关键不变量。


## 3. STE：forward 用三值，backward 更新高精度 master weight

round 几乎处处梯度为零。Straight-Through Estimator 用 `W + (Q(W)-W).detach()`，forward 数值等于量化权重，backward 把梯度近似传给 W。它是有偏估计，但实用。


In [ ]:
def ternary_ste(weights, group_size):  # 定义本节可复用的核心函数。
    _, _, quantized = ternary_quantize(weights, group_size)  # 计算并保存当前步骤的中间状态。
    return weights + (quantized - weights).detach()  # 返回当前分支计算出的结果。

# 权重 STE forward 等于分组反量化值，backward 更新每个浮点 master weight。
master = weight.clone().requires_grad_(True)  # 计算并保存当前步骤的中间状态。
ste_weight = ternary_ste(master, GROUP_SIZE)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(ste_weight, ternary_quantize(master, GROUP_SIZE)[2])  # 用受控断言验证关键不变量。
ste_weight.sum().backward()  # 计算并保存当前步骤的中间状态。
assert torch.allclose(master.grad, torch.ones_like(master))  # 用受控断言验证关键不变量。
assert master.dtype == torch.float32  # 用受控断言验证关键不变量。


## 4. 手写 BitLinear.forward：master、权重 code 与激活 scale 各司其职

模块参数仍是浮点 master weight；forward 对输入量化、对权重三值化，再做线性变换。bias 通常可省略或保持高精度。这里显式返回量化元数据用于测试，生产 API 可隐藏。


In [ ]:
class BitLinear(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_features, out_features, group_size):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if group_size <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("group_size 必须为正")  # 遇到非法合同立即显式失败。
        self.in_features, self.group_size = in_features, group_size  # 计算并保存当前步骤的中间状态。
        self.weight = nn.Parameter(torch.empty(out_features, in_features))  # 计算并保存当前步骤的中间状态。
        nn.init.normal_(self.weight, std=1 / math.sqrt(in_features))  # 计算并保存当前步骤的中间状态。

    def forward(self, inputs):  # 定义本节可复用的核心函数。
        if inputs.shape[-1] != self.in_features:  # 按当前条件选择后续控制路径。
            raise ValueError("输入末维与 in_features 不一致")  # 遇到非法合同立即显式失败。
        activation_codes, activation_scale, activation = quantize_activation(inputs)  # 计算并保存当前步骤的中间状态。
        weight_for_forward = ternary_ste(self.weight, self.group_size)  # 计算并保存当前步骤的中间状态。
        output = activation @ weight_for_forward.T  # 计算并保存当前步骤的中间状态。
        return output, activation_codes, activation_scale  # 返回当前分支计算出的结果。

# 主 forward 同时使用激活/权重 STE；输入与 master weight 都收到有限非零梯度。
layer = BitLinear(5, 3, GROUP_SIZE)  # 计算并保存当前步骤的中间状态。
train_input = x.clone().requires_grad_(True)  # 计算并保存当前步骤的中间状态。
bit_output, input_codes, input_scale = layer(train_input)  # 计算并保存当前步骤的中间状态。
bit_output.square().mean().backward()  # 计算并保存当前步骤的中间状态。
assert bit_output.shape == (4, 3)  # 用受控断言验证关键不变量。
assert layer.weight.grad is not None and torch.isfinite(layer.weight.grad).all()  # 用受控断言验证关键不变量。
assert train_input.grad is not None and torch.isfinite(train_input.grad).all()  # 用受控断言验证关键不变量。
assert (train_input.grad.norm(dim=-1) > 0).all()  # 用受控断言验证关键不变量。
assert list(dict(layer.named_parameters())) == ["weight"]  # 用受控断言验证关键不变量。


## 5. 整数点积等价：先算 code，再合并 scale

若 activation 每行 scale 为 `s_x`、权重共享 scale 为 `s_w`，则反量化点积等于 `int_dot * s_x*s_w`。真实实现用更宽 accumulator 防溢出，并按分组/通道处理 scale。


In [ ]:
def grouped_integer_linear(input_codes, input_scale, weight_codes, weight_scales, group_size):  # 定义本节可复用的核心函数。
    output = torch.zeros(input_codes.shape[0], weight_codes.shape[0], dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
    accumulators = []  # 计算并保存当前步骤的中间状态。
    for group_id, start in enumerate(range(0, input_codes.shape[-1], group_size)):  # 遍历输入元素以累积或检查结果。
        stop = min(start + group_size, input_codes.shape[-1])  # 计算并保存当前步骤的中间状态。
        accumulator = input_codes[:, start:stop].to(torch.int32) @ weight_codes[:, start:stop].to(torch.int32).T  # 计算并保存当前步骤的中间状态。
        output += accumulator.float() * input_scale * weight_scales[:, group_id, 0][None, :]  # 计算并保存当前步骤的中间状态。
        accumulators.append(accumulator)  # 计算并保存当前步骤的中间状态。
    return output, accumulators  # 返回当前分支计算出的结果。

# 整数 oracle 直接复原主 BitLinear.forward，而不是另造一个无关 reference。
weight_codes, grouped_scale, _ = ternary_quantize(layer.weight.detach(), layer.group_size)  # 计算并保存当前步骤的中间状态。
integer_reconstructed, accumulators = grouped_integer_linear(input_codes, input_scale, weight_codes, grouped_scale, layer.group_size)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(integer_reconstructed, bit_output.detach(), atol=1e-5)  # 用受控断言验证关键不变量。
assert all(accumulator.dtype == torch.int32 for accumulator in accumulators)  # 用受控断言验证关键不变量。
assert integer_reconstructed.shape == bit_output.shape  # 用受控断言验证关键不变量。


## 6. Base-3 位打包：理论 1.58 bit 需要真实编码才能兑现

把 -1/0/1 映射到 0/1/2，5 个 trit 可装进一个 byte，因为 `3^5=243<256`。这平均 1.6 bit/weight，接近信息下界；还需保存 scale、shape、尾部长度和对齐。


In [ ]:
def pack_trits(ternary_values):  # 定义本节可复用的核心函数。
    array = np.asarray(ternary_values, dtype=np.int8).reshape(-1)  # 计算并保存当前步骤的中间状态。
    if not np.isin(array, [-1, 0, 1]).all():  # 按当前条件选择后续控制路径。
        raise ValueError("只能打包 -1/0/1")  # 遇到非法合同立即显式失败。
    digits = (array + 1).tolist()  # 计算并保存当前步骤的中间状态。
    packed = bytearray()  # 计算并保存当前步骤的中间状态。
    for start in range(0, len(digits), 5):  # 遍历输入元素以累积或检查结果。
        value = sum(digit * (3 ** offset) for offset, digit in enumerate(digits[start:start + 5]))  # 计算并保存当前步骤的中间状态。
        packed.append(value)  # 计算并保存当前步骤的中间状态。
    return bytes(packed), len(digits)  # 返回当前分支计算出的结果。

def unpack_trits(packed, count):  # 定义本节可复用的核心函数。
    if not isinstance(count, int) or count < 0 or len(packed) != math.ceil(count / 5):  # 按当前条件选择后续控制路径。
        raise ValueError("packed 长度与 count 不一致")  # 遇到非法合同立即显式失败。
    raw_digits = []  # 计算并保存当前步骤的中间状态。
    for byte in packed:  # 遍历输入元素以累积或检查结果。
        if byte >= 3 ** 5:  # 按当前条件选择后续控制路径。
            raise ValueError("存在非规范 base-3 byte")  # 遇到非法合同立即显式失败。
        value = byte  # 计算并保存当前步骤的中间状态。
        for _ in range(5):  # 遍历输入元素以累积或检查结果。
            raw_digits.append(value % 3); value //= 3  # 计算并保存当前步骤的中间状态。
    if any(raw_digits[count:]):  # 按当前条件选择后续控制路径。
        raise ValueError("尾部未使用 trit 必须为规范零填充")  # 遇到非法合同立即显式失败。
    return np.array(raw_digits[:count], dtype=np.int8) - 1  # 返回当前分支计算出的结果。

def pack_checkpoint(weight_codes, scales, group_size, activation_bits=8):  # 定义本节可复用的核心函数。
    if weight_codes.ndim != 2 or group_size <= 0 or not 2 <= activation_bits <= 8:  # 按当前条件选择后续控制路径。
        raise ValueError("checkpoint 量化参数非法")  # 遇到非法合同立即显式失败。
    expected_scale_shape = (weight_codes.shape[0], math.ceil(weight_codes.shape[1] / group_size), 1)  # 计算并保存当前步骤的中间状态。
    if tuple(scales.shape) != expected_scale_shape or not torch.isfinite(scales).all() or not (scales > 0).all():  # 按当前条件选择后续控制路径。
        raise ValueError("checkpoint scale shape/value 与分组合同不一致")  # 遇到非法合同立即显式失败。
    packed, count = pack_trits(weight_codes.cpu().numpy())  # 计算并保存当前步骤的中间状态。
    return {  # 返回当前分支计算出的结果。
        "packed": packed, "count": count, "shape": tuple(weight_codes.shape),  # 执行当前语句以推进本节示例。
        "scales": scales.detach().clone(), "group_size": group_size,  # 执行当前语句以推进本节示例。
        "activation_bits": activation_bits, "layout": "out-channel-contiguous-v1",  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。

def restore_checkpoint(checkpoint):  # 定义本节可复用的核心函数。
    required = {"packed", "count", "shape", "scales", "group_size", "activation_bits", "layout"}  # 计算并保存当前步骤的中间状态。
    if not required <= set(checkpoint):  # 按当前条件选择后续控制路径。
        raise ValueError("checkpoint 缺少必要元数据")  # 遇到非法合同立即显式失败。
    shape, group_size = checkpoint["shape"], checkpoint["group_size"]  # 计算并保存当前步骤的中间状态。
    if not isinstance(shape, (tuple, list)) or len(shape) != 2 or not isinstance(group_size, int) or group_size <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("checkpoint shape/group_size 非法")  # 遇到非法合同立即显式失败。
    if checkpoint["layout"] != "out-channel-contiguous-v1" or not 2 <= checkpoint["activation_bits"] <= 8:  # 按当前条件选择后续控制路径。
        raise ValueError("checkpoint layout/activation_bits 不受支持")  # 遇到非法合同立即显式失败。
    expected_scale_shape = (shape[0], math.ceil(shape[1] / group_size), 1)  # 计算并保存当前步骤的中间状态。
    scales = checkpoint["scales"]  # 计算并保存当前步骤的中间状态。
    if math.prod(shape) != checkpoint["count"] or tuple(scales.shape) != expected_scale_shape:  # 按当前条件选择后续控制路径。
        raise ValueError("checkpoint count/scale shape 与权重布局不一致")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(scales).all() or not (scales > 0).all():  # 按当前条件选择后续控制路径。
        raise ValueError("checkpoint scale 必须为有限正数")  # 遇到非法合同立即显式失败。
    values = unpack_trits(checkpoint["packed"], checkpoint["count"])  # 计算并保存当前步骤的中间状态。
    return torch.from_numpy(values.reshape(shape)), scales.clone(), group_size, checkpoint["activation_bits"]  # 返回当前分支计算出的结果。

def packed_checkpoint_linear(checkpoint, activation_codes, activation_scale):  # 定义本节可复用的核心函数。
    weight_codes, scales, group_size, activation_bits = restore_checkpoint(checkpoint)  # 计算并保存当前步骤的中间状态。
    if activation_codes.dtype != torch.int8 or activation_scale.shape != (activation_codes.shape[0], 1):  # 按当前条件选择后续控制路径。
        raise ValueError("activation code/scale 布局非法")  # 遇到非法合同立即显式失败。
    qmax = 2 ** (activation_bits - 1) - 1  # 计算并保存当前步骤的中间状态。
    if activation_codes.abs().max() > qmax or not torch.isfinite(activation_scale).all() or not (activation_scale > 0).all():  # 按当前条件选择后续控制路径。
        raise ValueError("activation code/scale 数值非法")  # 遇到非法合同立即显式失败。
    return grouped_integer_linear(activation_codes, activation_scale, weight_codes, scales, group_size)[0]  # 返回当前分支计算出的结果。

# packed checkpoint 可直接恢复并执行整数推理；坏 scale/布局/byte/count 均 fail-closed。
checkpoint = pack_checkpoint(weight_codes, grouped_scale, layer.group_size, activation_bits=8)  # 计算并保存当前步骤的中间状态。
restored_codes, restored_scales, restored_group_size, restored_bits = restore_checkpoint(checkpoint)  # 计算并保存当前步骤的中间状态。
checkpoint_output = packed_checkpoint_linear(checkpoint, input_codes, input_scale)  # 计算并保存当前步骤的中间状态。
assert torch.equal(restored_codes, weight_codes.cpu()) and torch.equal(restored_scales, grouped_scale)  # 用受控断言验证关键不变量。
assert restored_group_size == layer.group_size and restored_bits == 8  # 用受控断言验证关键不变量。
assert torch.allclose(checkpoint_output, bit_output.detach(), atol=1e-5)  # 用受控断言验证关键不变量。

bad_scale = dict(checkpoint); bad_scale["scales"] = torch.ones(1)  # 计算并保存当前步骤的中间状态。
bad_layout = dict(checkpoint); bad_layout["layout"] = "unknown"  # 计算并保存当前步骤的中间状态。
bad_count = dict(checkpoint); bad_count["count"] -= 1  # 计算并保存当前步骤的中间状态。
fail_closed = 0  # 计算并保存当前步骤的中间状态。
for operation in (  # 遍历输入元素以累积或检查结果。
    lambda: pack_trits([0, 2]), lambda: unpack_trits(bytes([255]), 1),  # 执行当前语句以推进本节示例。
    lambda: restore_checkpoint(bad_scale), lambda: restore_checkpoint(bad_layout), lambda: restore_checkpoint(bad_count),  # 执行当前语句以推进本节示例。
):  # 执行当前语句以推进本节示例。
    try:  # 尝试执行可能失败的受控操作。
        operation()  # 执行当前语句以推进本节示例。
    except ValueError:  # 捕获预期异常并验证失败分支。
        fail_closed += 1  # 计算并保存当前步骤的中间状态。
assert fail_closed == 5  # 用受控断言验证关键不变量。


## 7. 误差与存储：必须把 scale/布局开销计入

三值误差可用输出相对误差与下游 loss 衡量。存储估计不能简单写参数数×1.58，还要计每组 scale、padding、索引和未量化 embedding/norm；训练还保留 master、梯度和优化器状态。


In [ ]:
def packed_storage_bytes(num_weights, group_size, scale_bytes=2):  # 定义本节可复用的核心函数。
    if num_weights < 0 or group_size <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("容量参数非法")  # 遇到非法合同立即显式失败。
    code_bytes = math.ceil(num_weights / 5)  # 计算并保存当前步骤的中间状态。
    scale_count = math.ceil(num_weights / group_size)  # 计算并保存当前步骤的中间状态。
    return code_bytes + scale_count * scale_bytes  # 返回当前分支计算出的结果。

# 估算使用与 forward 相同的分组合同；推理紧凑存储小于 FP16，训练 master 并未消失。
num_weights = 1_000_000  # 计算并保存当前步骤的中间状态。
ternary_bytes = packed_storage_bytes(num_weights, group_size=GROUP_SIZE)  # 计算并保存当前步骤的中间状态。
fp16_bytes = num_weights * 2  # 计算并保存当前步骤的中间状态。
relative_error = (dequantized - weight).norm() / weight.norm()  # 计算并保存当前步骤的中间状态。
assert ternary_bytes < fp16_bytes  # 用受控断言验证关键不变量。
assert 0 <= relative_error < 1  # 用受控断言验证关键不变量。
assert packed_storage_bytes(10, GROUP_SIZE) == 6  # 用受控断言验证关键不变量。


## 8. 发布门禁：从头训练 recipe 与 kernel layout 一起版本化

b1.58 的论文结果来自匹配的训练 recipe，不意味着任意 FP checkpoint 直接 ternary PTQ 仍保质量。manifest 绑定中心化/scale/阈值、激活位宽、分组、打包序、accumulator、kernel 与未量化层。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class BitNetArtifact:  # 定义承载本节状态与行为的数据结构。
    weight_scheme: str  # 执行当前语句以推进本节示例。
    activation_bits: int  # 执行当前语句以推进本节示例。
    group_size: int  # 执行当前语句以推进本节示例。
    packing: str  # 执行当前语句以推进本节示例。
    accumulator: str  # 执行当前语句以推进本节示例。
    trained_with_quantization: bool  # 执行当前语句以推进本节示例。

def artifact_hash(artifact):  # 定义本节可复用的核心函数。
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

# artifact 的 group_size 与真实 BitLinear.forward、整数 oracle 和容量估算完全一致。
artifact = BitNetArtifact("grouped-absmean-ternary-v2", 8, GROUP_SIZE, "base3-5trits-canonical-v2", "int32", True)  # 计算并保存当前步骤的中间状态。
digest = artifact_hash(artifact)  # 计算并保存当前步骤的中间状态。
assert artifact.trained_with_quantization  # 用受控断言验证关键不变量。
assert artifact.group_size == layer.group_size  # 用受控断言验证关键不变量。
assert len(digest) == 64  # 用受控断言验证关键不变量。
assert digest != artifact_hash(BitNetArtifact(artifact.weight_scheme, 8, 4, artifact.packing, "int32", True))  # 用受控断言验证关键不变量。


## 面试收束：从公式走到生产合同

建议用六步回答：目标与约束、张量/数据合同、核心公式、正确性反例、质量—成本评测、版本与回滚。Notebook 的小模型只证明机制和边界，不代表论文规模结果、真实 GPU kernel 加速或线上泛化。生产替换时仍应保留同一批 oracle，并补齐目标硬件 profiling、分布式一致性、数据 provenance、安全审计和灰度发布。

继续追问时要主动区分：训练期方法与已有 checkpoint 的后处理、理论 FLOPs 与 wall-clock、平均质量与关键 slice、可逆近似与不可逆状态、模型置信与校准后的决策概率。
